# Dataset Construction and Storage — GSE225845
This notebook constructs the tumor methylation matrix from **GSE225845** (Illumina HumanMethylationEPIC BeadChip, ~800k CpGs).  
The raw normalized β-values are imported in streaming mode and stored as an LZ4-compressed Parquet file to ensure reproducibility and schema consistency.

Unlike the *normal* and *normal-adjacent* cohorts — successfully converted to `Float32` — this dataset was retained in `Float64` due to the high number of CpG loci and the resulting memory allocation limits.

**Source: GEO accession GSE225845, platform Illumina HumanMethylation450 BeadChip**

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   POLITECNICO DI TORINO — MSc Mathematical Engineering           ║
# ║   THESIS: Advanced Study of Epigenetic Mechanisms in Neoplasms   ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Author:       Elisabetta Roviera (s328422)                       ║
# ║ Supervisors:  Dr. Sandro Gambino, Prof. Alfredo Benso            ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Notebook:     00-dataset-preparation-gse225845                   ║
# ║ Description:  Construction of the core methylation matrix        ║
#                 (GSE225845) — Parquet storage                      ║
# ║ Dataset(s):   GSE69914                                           ║
# ║ Repository:   github.com/elisabettaroviera/THESIS                ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Date: 09-Nov-2025 | Python 3.11.13                               ║
# ╚══════════════════════════════════════════════════════════════════╝


### Libraries

In [ ]:
!pip -q install polars pyarrow


In [ ]:
import polars as pl
import os
import gc


## 1. Read TXT → write Parquet (LZ4)

In [ ]:
# READ TXT (streaming) -> WRITE PARQUET LZ4¶
# 1. Input / output paths
print("1. Input / output paths - Start")
INPUT_TXT = "/kaggle/input/gse225845-tumors-normalized-betas-txt/GSE225845_tumors_normalized_betas.txt"
OUT_PAR = "/kaggle/working/GSE225845_tumor_lz4.parquet"

assert os.path.exists(INPUT_TXT), f"Input file not found: {INPUT_TXT}" print("1. Input / output paths - End")

# 2. Columns that must stay as strings
print("2. Columns that must stay as strings - Start")
STRING_COLS = ["accession_num", "basenames"]
print("2. Columns that must stay as strings - End")

# 3. Build a lazy CSV scan
# This does NOT load the whole file into memory.
print("3. Build a lazy CSV scan - Start")
lf = pl.scan_csv( INPUT_TXT, 
                 separator="\t", # TXT is tab-separated
                 has_header=True,
                 infer_schema_length=1000, # look at first N rows to guess types
                 null_values=["NA", "NaN", ""] )
print("3. Build a lazy CSV scan - End")

# 4. Cast dtypes lazily:
# - accession_num, basenames -> Utf8 (strings)
# - all other columns -> Float32 (beta values)
print("4. Cast dtypes lazily - Start")
present_string_cols = [c for c in STRING_COLS if c in lf.columns]

exprs = []

# Keep string columns as Utf8
for col in present_string_cols: exprs.append(pl.col(col).cast(pl.Utf8))

# All remaining columns (CpG probes) as Float32
exprs.append( pl.all() .exclude(present_string_cols) .cast(pl.Float32) )

lf = lf.with_columns(exprs)
print("4. Cast dtypes lazily - End")

# 5. Write to Parquet (streaming sink)
# This will stream the data in chunks and never hold the full table in RAM.
print("5. Write to Parquet - Start")
lf.sink_parquet( OUT_PAR, compression="lz4", statistics=True )

print(f"✅ Parquet saved to: {OUT_PAR}") print("5. Write to Parquet - End")

# 6. Quick sanity check (now we can read the Parquet normally)
print("6. Quick sanity check - Start")
par = pl.read_parquet(OUT_PAR)
print("Parquet shape:", par.shape)

if present_string_cols:
    print(par.select(present_string_cols).head())

print("Parquet dtypes (first 5):", par.dtypes[:5]) print("6. Quick sanity check - End")


## 2. Read TXT → write Parquet (LZ4) for Tumor

In [ ]:
# READ TXT (streaming) -> WRITE PARQUET LZ4

# TXT (tab)  →  Parquet RAW (LZ4)
INPUT_TXT = "/kaggle/input/gse225845-tumors-normalized-betas-txt/GSE225845_tumors_normalized_betas.txt"
OUT_PAR_RAW   = "/kaggle/working/GSE225845_tumor_lz4_float64.parquet"

print("1. Input / output paths - Start")
assert os.path.exists(INPUT_TXT), f"Input file not found: {INPUT_TXT}"
print(f"   Input:  {INPUT_TXT}")
print(f"   Output: {OUT_PAR_RAW}")
print("1. Input / output paths - End")

# Columns that must stay as strings (metadata)
STRING_COLS = ["accession_num", "basenames"]

print("2. Build a *lazy* TXT scan (streaming) - Start")
lf = pl.scan_csv(
    INPUT_TXT,
    separator="\t",          
    has_header=True,
    infer_schema_length=1000, 
    null_values=["NA", "NaN", ""],
    dtypes={                  
        "accession_num": pl.Utf8,
        "basenames": pl.Utf8,
    },
)
print("2. Build a *lazy* TXT scan (streaming) - End")

print("3. Write RAW Parquet (streaming sink) - Start")
lf.sink_parquet(
    OUT_PAR_RAW,
    compression="lz4",
    statistics=True,
)
print(f"✅ RAW Parquet saved to: {OUT_PAR_RAW}")
print("3. Write RAW Parquet (streaming sink) - End")

# Quick sanity check (eager, ma solo sul Parquet già compatto)
print("4. Quick sanity check - Start")
par = pl.read_parquet(OUT_PAR_RAW)
print("   Parquet shape:", par.shape)
print("   First 3 columns:", par.columns[:3])
print("   Dtypes (first 5):", par.dtypes[:5])
print("4. Quick sanity check - End")
